# Tensor smoothing & full-gradient inverse design

> Two accuracy/optimization features: **Kottke** subpixel smoothing for tilted
> interfaces, and **full `dS/dp`** gradients (every effect, not just the
> effective indices) for inverse design.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import meow as mw

jax.config.update("jax_enable_x64", True)
WL = 1.55

## 1. Kottke (normal-projected) subpixel smoothing

The default `"axis"` smoothing picks arithmetic *or* harmonic averaging per field
component from the interface's grid alignment - only first-order for tilted /
curved interfaces. `smoothing_method="kottke"` instead computes the interface
**normal** (from a fill-fraction gradient) and applies the normal-projected
effective permittivity $\varepsilon_{ii} = \varepsilon_\parallel + (\varepsilon_\perp - \varepsilon_\parallel)\,n_i^2$
- continuous in the interface orientation.

We test it by **rotation invariance**: a square waveguide's fundamental $n_\mathrm{eff}$
is the same axis-aligned or rotated, so the rotated-vs-reference error isolates
the smoothing quality at the (staircased) tilted edges.

In [ ]:
def square_neff(rot_deg, method, half=0.3, n_core=3.0, npx=151):
    th = np.deg2rad(rot_deg)
    rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    poly = np.array([(-half, -half), (half, -half), (half, half), (-half, half)]) @ rot.T
    core = mw.Structure(
        material=mw.IndexMaterial(name="core", n=n_core),
        geometry=mw.Prism(poly=poly, h_min=-1.0, h_max=2.0, axis="z"), mesh_order=5)
    clad = mw.Structure(
        material=mw.IndexMaterial(name="clad", n=1.444),
        geometry=mw.Box(x_min=-1.5, x_max=1.5, y_min=-1.5, y_max=1.5, z_min=-1, z_max=2),
        mesh_order=10)
    mesh = mw.Mesh2D(x=np.linspace(-1.5, 1.5, npx), y=np.linspace(-1.5, 1.5, npx))
    cell = mw.create_cells([core, clad], mesh, [1.0], z_min=0.0)[0]
    cs = mw.CrossSection.from_cell(cell=cell, env=mw.Environment(wl=WL),
                                   smoothing_method=method)
    return float(np.real(mw.compute_modes(cs, num_modes=1)[0].neff))


ref = square_neff(0.0, "axis")  # axis-aligned reference (edges aligned, ~exact)
angles = [10, 20, 30, 40, 50, 60, 70, 80]
err_axis = [abs(square_neff(a, "axis") - ref) for a in angles]
err_kottke = [abs(square_neff(a, "kottke") - ref) for a in angles]
print(f"mean error  axis={np.mean(err_axis):.2e}  kottke={np.mean(err_kottke):.2e}  "
      f"-> {np.mean(err_axis)/np.mean(err_kottke):.1f}x lower")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogy(angles, err_axis, "o-", label="axis smoothing")
ax.semilogy(angles, err_kottke, "s-", label="kottke smoothing")
ax.set(xlabel="waveguide rotation [deg]",
       ylabel=r"$|n_{eff} - n_{eff}^{ref}|$",
       title="Tilted-interface error: Kottke vs axis smoothing")
ax.grid(visible=True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()

Kottke is markedly closer to the rotation-invariant reference at every tilt
angle, so for a given accuracy you can use a coarser grid. It stays
diagonal-anisotropic, so the default solver handles it (the full tensor's
off-diagonal needs the tensorial solver / `tidy3d-extras`). Use it via
`CrossSection.from_cell(..., smoothing_method="kottke")`; the default remains
`"axis"`.

## 2. Full `dS/dp`: gradients through the whole solve

`make_differentiable_neffs` (notebook 07) gives a *cheap* gradient that is exact
for objectives mediated by the propagation constants. For the **complete**
`dS/dp` - including mode-overlap / interface (mode-mismatch) sensitivities -
`make_differentiable_objective` wraps a real, gauge-invariant figure of merit
(transmission, splitting ratio, loss) as a `jax.custom_vjp` whose backward
differences the *whole* solve. It composes with `jax.grad`, at the cost of
`2 N_params` re-solves.

> Why a real FoM and not the complex S: each mode solve has an arbitrary global
> phase, so the complex S-matrix is gauge-inconsistent across re-solves; only
> gauge-invariant powers are smooth in the parameters. (An *analytic* overlap
> adjoint via truncated guided modes is inaccurate for high-contrast
> waveguides - the overlap change is dominated by radiation modes outside the
> computed basis - so the robust exact route is to difference the full solve.)

Here we maximize the transmission of a butt-coupled width step by tuning the
input width - a mode-mismatch (overlap-dominated) objective.

In [ ]:
def transmission(params):
    w0 = float(params[0])
    widths = (w0, 0.9)
    mesh = mw.Mesh2D(x=np.linspace(-1.2, 1.2, 81), y=np.linspace(-0.5, 0.72, 41))
    structs = [mw.Structure(
        material=mw.IndexMaterial(name="si", n=3.45),
        geometry=mw.Box(x_min=-w/2, x_max=w/2, y_min=0, y_max=0.22,
                        z_min=2*i, z_max=2*(i+1)))
        for i, w in enumerate(widths)]
    cells = [mw.Cell(structures=structs, mesh=mesh, z_min=2*i, z_max=2*(i+1))
             for i in range(2)]
    env = mw.Environment(wl=WL)
    modes = [mw.compute_modes(mw.CrossSection.from_cell(cell=c, env=env), num_modes=1)
             for c in cells]
    s, _ = mw.compute_s_matrix(modes, cells=cells)
    return float(np.abs(np.asarray(s)[1, 0]) ** 2)


fom = mw.make_differentiable_objective(transmission, shape=(), step=5e-3)
value_and_grad = jax.value_and_grad(fom)

w = jnp.array([0.30])
lr, hist = 0.05, []
for _ in range(7):
    t, g = value_and_grad(w)
    hist.append((float(w[0]), float(t)))
    w = jnp.clip(w + lr * g, 0.2, 0.88)  # ascend transmission
hist = np.array(hist)
print(f"width {hist[0,0]:.3f} -> {float(w[0]):.3f} um,  "
      f"|T|^2 {hist[0,1]:.4f} -> {float(fom(w)):.4f}")

In [ ]:
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4))
axa.plot(hist[:, 0], "o-")
axa.set(xlabel="iteration", ylabel="input width [um]", title="design parameter")
axb.plot(hist[:, 1], "o-", color="C2")
axb.set(xlabel="iteration", ylabel="transmission |T|$^2$",
        title="gradient ascent on transmission (full dS/dp)")
fig.tight_layout()

The gradient came through the full solve, so it includes the mode-mismatch
(overlap) sensitivity that drives this butt-coupling objective - which the
cheap neff-only gradient cannot see. For many design parameters prefer the cheap
`make_differentiable_neffs` where the objective is propagation-mediated, and use
`make_differentiable_objective` when overlap/mismatch effects matter.

## Summary

- `smoothing_method="kottke"` - normal-projected tensor smoothing; lower error
  for tilted/curved interfaces (coarser grids for the same accuracy).
- `make_differentiable_objective` - exact full `dS/dp` for a gauge-invariant FoM
  via `jax.custom_vjp` (differences the whole solve; `2 N_params` re-solves).
- See `examples/benchmarks/performance_features.py` for timing of these plus the
  sparse solver and the smoothing cache.